# 03 · Participant-level Student Baseline — No KD

This is the clean **hard-label-only** student baseline.

- TRAIN = 107 participants
- DEV = 34 participants, 440 excluded
- one training sample = one participant
- aligned Participant-only audio/text segments
- checkpoint selection = DEV-34 participant macro-F1
- decision threshold fixed at 0.5
- teacher targets are not loaded
- TEST is not prepared or opened

Outputs also include parameters, checkpoint size, profiled GFLOPs, latency, throughput, and peak CUDA memory.


In [ ]:
from google.colab import drive,userdata
drive.mount('/content/drive')

from pathlib import Path
import base64,subprocess,sys,json
REPO='engrkhubabahmad/reliability-aware-depression-kd'
ROOT=Path('/content/reliability-aware-depression-kd')
token=userdata.get('GITHUB_TOKEN'); assert token,'Add GITHUB_TOKEN to Colab Secrets'
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git=['git','-c',f'http.extraHeader=AUTHORIZATION: basic {auth}']
if not (ROOT/'.git').exists(): subprocess.run(git+['clone',f'https://github.com/{REPO}.git',str(ROOT)],check=True)
subprocess.run(git+['-C',str(ROOT),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','main'],check=True)
subprocess.run(git+['-C',str(ROOT),'pull','--ff-only','origin','main'],check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-colab.txt')])
print('main:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


## 1. Prepare/resume participant features

Feature extraction is cached per participant in Drive. Rerunning reuses completed participant caches.


In [ ]:
DATA=Path('/content/drive/MyDrive/DAIC_WOZ')
FEAT=DATA/'processed'/'participant_student'/'v1'

subprocess.run([
 sys.executable,'-m','scripts.students.participant_student.prepare',
 '--daic-root',str(DATA),'--output',str(FEAT)
],cwd=ROOT,check=True)


## 2. Inspect preparation protocol


In [ ]:
summary=json.loads((FEAT/'feature_summary.json').read_text())
print(json.dumps(summary,indent=2))
assert summary['train_participants']==107
assert summary['dev_participants']==34
assert summary['participant_440_excluded'] is True
assert summary['test_opened'] is False and summary['test_prepared'] is False


## 3. Train No-KD participant student

Default participant batch size is 16. No threshold search is performed.


In [ ]:
OUT=DATA/'experiments'/'students'/'participant_v1'/'no_kd'/'seed_103'

subprocess.run([
 sys.executable,'-m','scripts.students.participant_student.train_baseline',
 '--features',str(FEAT),'--output',str(OUT),
 '--seed','103','--batch-size','16',
 '--epochs','120','--patience','20',
 '--lr','3e-4','--weight-decay','1e-4'
],cwd=ROOT,check=True)


## 4. Review DEV-34 and complexity


In [ ]:
import pandas as pd
metrics=json.loads((OUT/'metrics.json').read_text())
complexity=json.loads((OUT/'model_complexity.json').read_text())
protocol=json.loads((OUT/'training_protocol.json').read_text())

print('DEV-34')
print(json.dumps(metrics['dev34'],indent=2))
print('\nMODEL COMPLEXITY')
print(json.dumps(complexity,indent=2))
print('\nPROTOCOL')
print(json.dumps(protocol,indent=2))

display(pd.read_csv(OUT/'dev_predictions.csv'))
assert protocol['teacher_targets_loaded'] is False
assert protocol['test_opened'] is False


## 5. Stop here

Do not open TEST. After accepting the No-KD baseline, run notebook **04_kd.ipynb** for the teacher-distilled variants.
